In [ ]:
# OmniVoice's own lockfile, from PyTorch's cu128 index for torch; torchvision at
# the version that belongs to that torch so transformers can import beside it.
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip -q install "transformers==5.3.0" "accelerate==1.13.0" "webdataset==1.0.2" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.7.2" "tensorboardX==2.6.4" "soundfile==0.13.1" "librosa==0.11.0" "pydub==0.25.1" 2>&1 | tail -3
!pip -q install --no-deps "git+https://github.com/k2-fsa/OmniVoice@08be0b4ccbac3e13e374e86fbfead4b4cac343e2" 2>&1 | tail -3


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def captured(command):
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.8.0+cu128', 'torchaudio': '2.8.0+cu128', 'transformers': '5.3.0', 'accelerate': '1.13.0', 'webdataset': '1.0.2', 'safetensors': '0.7.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.7.2', 'tensorboardX': '2.6.4', 'soundfile': '0.13.1', 'librosa': '0.11.0', 'pydub': '0.25.1', 'numpy': '2.4.3', 'triton': '3.4.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'accelerate', 'webdataset', 'safetensors', 'tokenizers', 'huggingface-hub')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {name: {"official": want, "installed": installed(name),
                "matches": matches(installed(name), want), "pinned_here": name in PINNED}
         for name, want in OFFICIAL.items()}
direct = json.loads(importlib.metadata.distribution("omnivoice").read_text("direct_url.json") or "{}")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "bf16_supported_including_emulation": torch.cuda.is_bf16_supported(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "omnivoice": {"version": installed("omnivoice"),
                  "commit": (direct.get("vcs_info") or {}).get("commit_id"), "url": direct.get("url")},
    "other_packages": {name: installed(name) for name in ("torchvision", "ipykernel", "papermill", "scipy")},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:16} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("omnivoice", ENVIRONMENT_RECORD["omnivoice"])
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"
assert ENVIRONMENT_RECORD["omnivoice"]["commit"] == "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", ENVIRONMENT_RECORD["omnivoice"]
assert torch.cuda.device_count() >= 2, "the official recipe trains on two GPUs - use T4 x2"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GiB free under /tmp")

INPUT = Path("/kaggle/input")
found = sorted(INPUT.rglob("repo/scripts/omnivoice_adapt.py"))
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo with scripts/omnivoice_adapt.py under /kaggle/input")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
# Kaggle unpacks the .gz when it builds the dataset; either form is checked
# against the hash the split builder recorded for its decompressed content.
MANIFESTS = REPO / "eval/manifests"
SPLITS = next((p for p in (MANIFESTS / "adaptation_splits_v1.jsonl.gz", MANIFESTS / "adaptation_splits_v1.jsonl")
               if p.is_file()), None)
assert SPLITS is not None, "the dataset predates the adaptation splits"
import gzip  # noqa: E402

content = gzip.open(SPLITS).read() if SPLITS.suffix == ".gz" else SPLITS.read_bytes()
expected = json.loads((MANIFESTS / "adaptation_splits_v1.summary.json").read_text())["records_sha256"]
assert hashlib.sha256(content).hexdigest() == expected, "the split file is not the one the builder wrote"
print("splits:", SPLITS.name, "sha256 matches the summary")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-omnivoice-probe-v1"
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import Gates, report_group, run_bounded  # noqa: E402

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
WORK = Path("/tmp/adapt")
WORK.mkdir(parents=True, exist_ok=True)
RECORDS = Path("/kaggle/working/records")
RECORDS.mkdir(exist_ok=True)


In [ ]:
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

BLOBS = {'examples/config/train_config_finetune.json': 'f80f573b158b', 'examples/config/train_config_finetune_sdpa.json': '4b0a543e23be', 'examples/config/data_config_finetune.json': '7b106dd96155'}


def blob_id(data):
    return hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data).hexdigest()


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


ARTIFACTS = {}
if not GATES.skip("download", ("runner",)):
    try:
        start = time.time()
        BASE = Path(snapshot_download("k2-fsa/OmniVoice", revision="c5fdb5ccb189668d56333f77ba2629f4cd7535f4", local_dir="/tmp/omnivoice-base"))
        TOKENIZER = Path(snapshot_download("eustlb/higgs-audio-v2-tokenizer", revision="528e871c2a26c4f0f7773b9754e2e1acae20899d",
                                           local_dir="/tmp/higgs-audio-v2-tokenizer"))
        hashes = {"tokenizer": sha256(TOKENIZER / "model.safetensors"),
                  "bundled_with_base": sha256(BASE / "audio_tokenizer/model.safetensors")}
        assert set(hashes.values()) == {"fe7c5e8785e0a05833e1bfc3e002ec7f55af21e306b2e7154a448c1f54ccfb0d"}, hashes
        OFFICIAL_CONFIGS = {}
        for name, blob in BLOBS.items():
            with urllib.request.urlopen(f"https://raw.githubusercontent.com/k2-fsa/OmniVoice/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/{name}") as response:
                data = response.read()
            assert blob_id(data).startswith(blob), (name, blob_id(data))
            target = WORK / "official" / Path(name).name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            OFFICIAL_CONFIGS[Path(name).name] = target
        CV_DIR, NEY_DIR = Path("/tmp/cv26"), Path("/tmp/neyshekar")
        for repo, revision, files, folder in (("pymmdrza/Common-Voice-Speech-26.0-Persian-Clean", "142e9bb40b56d9491faa0ec0987ee8d408bd9c05", ('data/train-00000-of-00034.parquet', 'data/validation-00000-of-00001.parquet'), CV_DIR),
                                              ("shekar-ai/neyshekar", "30696ada746a66bb3047c984744b5ecfd60fe4e9", ('data/train-00000-of-00022.parquet', 'data/validation-00000-of-00004.parquet'), NEY_DIR)):
            for name in files:
                hf_hub_download(repo, name, repo_type="dataset", revision=revision, local_dir=str(folder))
        ARTIFACTS = {"base": str(BASE), "tokenizer": str(TOKENIZER), "tokenizer_sha256": hashes,
                     "configs": {k: str(v) for k, v in OFFICIAL_CONFIGS.items()}}
        GATES.record("download", True, f"{(time.time() - start) / 60:.1f} min; tokenizer hashes agree; "
                     f"{len(OFFICIAL_CONFIGS)} official configs match their blob ids")
    except Exception as exc:  # noqa: BLE001 - the gate records it
        GATES.record("download", False, f"{type(exc).__name__}: {exc}")


In [ ]:
SLICE = {}
if not GATES.skip("data", ("download",)):
    ok = True
    for split, limit in (("train", 700), ("valid", 80)):
        summary_path = RECORDS / f"data_{split}.json"
        run = run_bounded([sys.executable, str(REPO / "scripts/prepare_adaptation_data.py"),
                           "--splits", str(SPLITS), "--split", split, "--cv-dir", str(CV_DIR),
                           "--neyshekar-dir", str(NEY_DIR), "--out", str(WORK / "data" / split),
                           "--limit", str(limit), "--summary", str(summary_path)],
                          timeout=1800, log=LOGS / f"data_{split}.log")
        RUNS[f"data_{split}"] = run.record()
        print(run.summary())
        if not run.ok or not summary_path.is_file():
            ok = False
            continue
        SLICE[split] = json.loads(summary_path.read_text())
        ok = ok and sum(SLICE[split]["written"].values()) > 0 and not any(SLICE[split]["left_out"].values())
    GATES.record("data", ok, json.dumps({k: v["written"] for k, v in SLICE.items()}))


In [ ]:
TOKENS = WORK / "tokens"
TOKEN_FACTS = {}
if not GATES.skip("tokenize", ("data",)):
    ok = True
    for split in ("train", "valid"):
        source = SLICE[split]["jsonl"][split]
        start = time.time()
        run = run_bounded([sys.executable, "-m", "omnivoice.scripts.extract_audio_tokens",
                           "--input_jsonl", source,
                           "--tar_output_pattern", str(TOKENS / split / "audios/shard-%06d.tar"),
                           "--jsonl_output_pattern", str(TOKENS / split / "txts/shard-%06d.jsonl"),
                           "--tokenizer_path", ARTIFACTS["tokenizer"], "--nj_per_gpu", "3", "--shuffle", "True"],
                          timeout=3600, log=LOGS / f"tokenize_{split}.log")
        RUNS[f"tokenize_{split}"] = run.record()
        print(run.summary())
        manifest = TOKENS / split / "data.lst"
        errors = TOKENS / split / "errors.jsonl"
        lines = manifest.read_text().splitlines() if manifest.is_file() else []
        samples = sum(int(line.split()[2]) for line in lines)
        seconds = sum(float(line.split()[3]) for line in lines)
        failed = len(errors.read_text().splitlines()) if errors.is_file() else 0
        TOKEN_FACTS[split] = {"shards": len(lines), "samples": samples, "audio_seconds": round(seconds, 1),
                              "failed": failed, "wall_seconds": round(time.time() - start, 1),
                              "realtime_factor": round(seconds / max(time.time() - start, 1e-6), 1)}
        ok = ok and run.ok and samples > 0 and failed == 0
    if ok:
        import io
        import tarfile

        import numpy as np
        first = sorted((TOKENS / "train" / "audios").glob("*.tar"))[0]
        with tarfile.open(first) as archive:
            member = next(m for m in archive.getmembers() if m.name.endswith(".npy"))
            tokens = np.load(io.BytesIO(archive.extractfile(member).read()))
        TOKEN_FACTS["example"] = {"member": member.name, "shape": list(tokens.shape), "dtype": str(tokens.dtype),
                                  "max": int(tokens.max()), "min": int(tokens.min())}
        ok = tokens.ndim == 2 and tokens.shape[0] == 8 and tokens.dtype == np.int16 and 0 <= tokens.min() and tokens.max() < 1024
    (RECORDS / "tokens.json").write_text(json.dumps(TOKEN_FACTS, indent=2))
    GATES.record("tokenize", ok, json.dumps(TOKEN_FACTS))


In [ ]:
DATA_CONFIG = WORK / "data_config.json"
STEP_RECORDS = {}


def attempt(name, config_path, steps, memory="official", timeout=2400, evaluate=False, extra=()):
    record = RECORDS / f"{name}.json"
    # The official launch line from run_finetune.sh, with the wrapper in place of -m omnivoice.cli.train.
    command = ["accelerate", "launch", "--gpu_ids", "0,1", "--num_processes", "2",
               str(REPO / "scripts/omnivoice_adapt.py"),
               "--train_config", str(config_path), "--data_config", str(DATA_CONFIG),
               "--output_dir", str(WORK / "runs" / name), "--set", f"init_from_checkpoint={ARTIFACTS['base']}",
               "--stop-after-steps", str(steps), "--memory", memory, "--record", str(record), *extra]
    if evaluate:
        command.append("--evaluate-once")
    run = run_bounded(command, timeout=timeout, log=LOGS / f"{name}.log", monitor_every=300)
    RUNS[name] = run.record()
    print(run.summary())
    ranks = [json.loads(p.read_text()) for p in sorted(RECORDS.glob(f"{name}.rank*.json"))]
    tail = run.log.read_text(errors="replace").splitlines()[-25:] if run.log.is_file() else []
    oom = any("out of memory" in line.lower() or "OutOfMemoryError" in line for line in tail)
    STEP_RECORDS[name] = {"ok": run.ok, "outcome": run.outcome, "oom": oom, "ranks": ranks, "log_tail": tail}
    return STEP_RECORDS[name]


def summarise(result):
    if not result["ranks"]:
        return f"{result['outcome']}; out of memory: {result['oom']}; last line: {(result['log_tail'] or [''])[-1][:300]}"
    return "; ".join(f"rank {r['rank']}: {r['optimizer_steps']} steps, {r['mean_step_seconds_after_two']} s/step, "
                     f"peak {r['peak_allocated_gib']} GiB allocated / {r['peak_reserved_gib']} reserved of "
                     f"{r['capacity_gib']}, losses {r['train_losses'][:4]}" for r in result["ranks"])


if not GATES.skip("flex", ("tokenize",)):
    DATA_CONFIG.write_text(json.dumps({"train": [{"manifest_path": [str(TOKENS / "train/data.lst")]}],
                                        "dev": [{"manifest_path": [str(TOKENS / "valid/data.lst")]}]}, indent=2))
    result = attempt("flex", ARTIFACTS["configs"]["train_config_finetune.json"], 4, timeout=1800)
    GATES.record("flex", result["ok"], summarise(result))

if not GATES.skip("sdpa", ("tokenize",)):
    result = attempt("sdpa", ARTIFACTS["configs"]["train_config_finetune_sdpa.json"], 8, evaluate=True)
    GATES.record("sdpa", result["ok"], summarise(result))

MEMORY_MODE = None
if GATES.status.get("sdpa", (False, ""))[0]:
    MEMORY_MODE = "official"
elif STEP_RECORDS.get("sdpa", {}).get("oom"):
    result = attempt("sdpa_memory", ARTIFACTS["configs"]["train_config_finetune_sdpa.json"], 8,
                     memory="checkpointing+bucket-view", evaluate=True)
    GATES.record("sdpa-memory", result["ok"], summarise(result))
    if result["ok"]:
        MEMORY_MODE = "checkpointing+bucket-view"
        # A test-only batch that fits both ways: the memory techniques must leave
        # the first two losses where the official path puts them.
        small = json.loads(Path(ARTIFACTS["configs"]["train_config_finetune_sdpa.json"]).read_text())
        small["batch_tokens"] = 2048
        small_path = WORK / "official" / "equivalence_test_only.json"
        small_path.write_text(json.dumps(small, indent=2))
        plain = attempt("equivalence_official", small_path, 2)
        tricks = attempt("equivalence_memory", small_path, 2, memory="checkpointing+bucket-view")
        losses = [[r["train_losses"][:2] for r in x["ranks"]] for x in (plain, tricks)]
        same = bool(plain["ranks"]) and bool(tricks["ranks"]) and all(
            abs(a - b) <= 5e-3 * max(abs(a), 1.0)
            for pr, tr in zip(losses[0], losses[1]) for a, b in zip(pr, tr))
        GATES.record("equivalence", same, f"first two losses per rank, official vs memory: {losses}")
        if not same:
            MEMORY_MODE = None


In [ ]:
BUDGET = {}
if MEMORY_MODE is None:
    GATES.record("budget", False, "no configuration of the full recipe ran on this hardware")
else:
    name = "sdpa" if MEMORY_MODE == "official" else "sdpa_memory"
    ranks = STEP_RECORDS[name]["ranks"]
    per_step = max(r["mean_step_seconds_after_two"] for r in ranks)
    train_rate = TOKEN_FACTS["train"]["realtime_factor"]
    evaluation = [r.get("evaluation_once") or {} for r in ranks]
    eval_seconds = max((e.get("seconds") or 0) for e in evaluation)
    valid_hours_slice = sum(SLICE["valid"]["hours"].values())
    eval_full = eval_seconds * (3.675 / max(valid_hours_slice, 1e-6))
    training_hours = 5000 * per_step / 3600
    tokenise_hours = 127.85 * 3600 / max(train_rate, 1e-6) / 3600
    evals_hours = 10 * eval_full / 3600
    total = training_hours + tokenise_hours + evals_hours
    BUDGET = {"memory_mode": MEMORY_MODE, "seconds_per_step": per_step,
              "training_hours": round(training_hours, 2), "tokenising_hours": round(tokenise_hours, 2),
              "evaluation_hours": round(evals_hours, 2), "total_hours": round(total, 2),
              "sessions_of_12h": int(-(-total // (12.0 - 1))),
              "weekly_quota_hours": 30.0,
              "peaks_gib": {f"rank {r['rank']}": r["peak_reserved_gib"] for r in ranks},
              "capacity_gib": ranks[0]["capacity_gib"]}
    fits = total <= 0.85 * 30.0
    GATES.record("budget", fits, json.dumps(BUDGET))
(RECORDS / "budget.json").write_text(json.dumps(BUDGET, indent=2))


In [ ]:
destination = Path("/kaggle/working/probe")
destination.mkdir(exist_ok=True)
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
(destination / "steps.json").write_text(json.dumps(STEP_RECORDS, indent=2))
(destination / "artifacts.json").write_text(json.dumps(ARTIFACTS, indent=2))
shutil.copy2("/kaggle/working/environment.json", destination / "environment.json")
for path in RECORDS.glob("*.json"):
    shutil.copy2(path, destination / path.name)
print("gates:", json.dumps(GATES.summary(), indent=1))
